[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_5_NonHomogeneous_PDEs_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 📡 اللاب 5: المؤثرات غير المتجانسة وتلاشي الإشارة والبطارية
### *غرفة التحكم: شبكة بث إشارات الملاحة واضمحلال الطاقة*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الباب الثاني (Chapter 2)، الأقسام 2.1–2.3 (صفحات 46–60)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $(a D_x + b D_y + c)z = 0$ | "مؤثر خطي غير متجانس يجمع بين الحركة والتلاشي الأسي." | `linear_factor(a, b, c)` |
| $z = e^{-\frac{c}{a}x}\Phi(ay - bx)$ | "موجة متحركة مع غلاف اضمحلال أسي للطاقة $e^{kx}$." | `np.exp(-c*X/a) * phi` |
| $(F_1)(F_2)z = 0 \implies z = z_1 + z_2$ | "مبدأ التراكب (Superposition) لجمع إشارات أبراج الاتصال." | `Z_total = Z1 + Z2` |

---
## 1. المعنى البسيط: "الصوت الذي يتلاشى مع المسافة"
عندما تنادي بصوت عالٍ في وادٍ فسيح، ينتقل صوتك كموجة متحركة، لكن شدته تضعف تدريجياً كلما ابتعدت عن المصدر.
* هذا التلاشي الأسي يصفه الحد الثابت $c$ في المؤثر $(a D_x + b D_y + c)$.
* **المؤثرات القابلة للتحليل (Reducible Operators):** تمكننا من تجزئة معادلة معقدة متعددة الأبعاد إلى حاصل ضرب عوامل خطية بسيطة يسهل حل كل منها على حدة!

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** لفهم الباب الثاني من المنهج، وكيف نميز بين المؤثر القابل للتحليل (Reducible) والمؤثر غير القابل للتحليل (Irreducible).
* **ماذا تفيد مركبة المريخ؟** عند استقبال إشارات الملاحة من برجي بث لاسلكي على سطح المريخ، تتلاشى طاقة الإشارة وتستهلك بطارية المركبة بمعدل أسي $e^{-x}$ مع زيادة مسافة القيادة $x$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab5

%matplotlib inline
print("تم تهيئة بيئة اللاب 5 بنجاح! جاهز لحساب المؤثرات غير المتجانسة.")


---
## 3. الوصفة الرياضية وقانون العوامل الخطية (Reducible Factors Theorem)

لأي عامل خطي غير متجانس على الصورة:
$$(a D_x + b D_y + c) z = 0$$
إذا كان $a \neq 0$، فإن الحل العام يعطى بالقانون المباشر (صفحة 47 بالمذكرة):
$$z = e^{-\frac{c}{a}x} \Phi(a y - b x)$$

إذا كان المؤثر يتكون من حاصل ضرب عاملين:
$$(a_1 D_x + b_1 D_y + c_1)(a_2 D_x + b_2 D_y + c_2) z = 0$$
فإن الحل العام هو مجموع الحلين (مبدأ التراكب Superposition):
$$z = e^{-\frac{c_1}{a_1}x} \Phi_1(a_1 y - b_1 x) + e^{-\frac{c_2}{a_2}x} \Phi_2(a_2 y - b_2 x)$$


---
## 4. 📝 تحدي الورقة والقلم 1 (مسألة الباب الثاني من المنهج)

> 💡 *(إذا واجهتك صعوبة، راجع `INSTRUCTOR_ANSWER_KEY.md` القسم 5.1).*

**المسألة:** أوجد الحل العام للمعادلة القابلة للتحليل:
$$(2 D_x - 3 D_y + 1)(D_x + 2 D_y - 2)z = 0$$
* العامل الأول: $a_1 = 2, b_1 = -3, c_1 = 1 \implies z_1 = e^{-x/2} \Phi_1(2y + 3x)$.
* العامل الثاني: $a_2 = 1, b_2 = 2, c_2 = -2 \implies z_2 = e^{2x} \Phi_2(y - 2x)$.

اكتب صيغة الحل العام المجموع أدناه:


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ###
student_solution = "exp(-x/2)*Phi1(2y + 3x) + exp(2x)*Phi2(y - 2x)"
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_1(student_solution)


---
## 5. 🚀 تحدي جسر مركبة المريخ (معدل استهلاك طاقة البطارية)

طاقة بطارية المركبة تتلاشى مع المسافة $x$ وفقاً للمعادلة:
$$(D_x + 2 D_y + 1)z = 0$$

**على ورقتك:**
1. المعاملات: $a = 1, b = 2, c = 1$.
2. احسب معدل التخميد الأسي $k$ في $e^{k x}$ من الصيغة $-\frac{c}{a}$.

ما هي القيمة العددية لمعدل التخميد $k$؟ أدخلها أدناه:


In [ ]:
# خلية تقييم: معدل تخميد البطارية
### START CODE HERE ###
damping_rate = -1.0
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab5.check_paper_challenge_2(damping_rate)


---
## 6. 💻 التطبيق البرمجي ببايثون: مستخرج معاملات العوامل الخطية

سنكتب دالة برمجية تستخرج المعاملات $(a, b, c)$ من أي نص عامل خطي مثل `"2*D_x - 3*D_y - 5"`:


In [ ]:
# دالة تقييم برمجية: parse_linear_factor
def parse_linear_factor(factor_str: str) -> tuple[float, float, float]:
    # تستخرج المعاملات a, b, c من نص العامل الخطي
    ### START CODE HERE ###
    import re
    ma = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_x', factor_str)
    mb = re.search(r'([+-]?\s*\d*)\s*\*?\s*D_y', factor_str)
    mc = re.search(r'([+-]?\s*\d+)\s*$', factor_str.replace(" ", ""))
    
    def to_float(val, default):
        if not val or val.strip() in ['+', '']: return default
        if val.strip() == '-': return -default
        return float(val.replace(" ", ""))

    a = to_float(ma.group(1), 1.0) if ma else 0.0
    b = to_float(mb.group(1), 1.0) if mb else 0.0
    c = float(mc.group(1)) if mc else 0.0
    ### END CODE HERE ###
    return a, b, c

# تشغيل الاختبار الآلي:
test_lab5.test_linear_factor_evaluator(parse_linear_factor)


---
## 7. 🎨 الرؤية البصرية للفيزياء: الإشارة المتلاشية أسياً
شغّل الخلية لرؤية كيف يتلاشى سطح الجهد أسياً مع المسافة $x$ وفقاً لـ $z = e^{-0.5x} \sin(y - 2x)$:


In [ ]:
x = np.linspace(0, 5, 50)
y = np.linspace(-3, 3, 50)
X, Y = np.meshgrid(x, y)
Z = np.exp(-0.5 * X) * np.sin(Y - 2.0 * X)

fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='plasma', alpha=0.85)
ax.set_title("Damped Signal Surface $z = e^{-0.5x} \\sin(y - 2x)$", fontsize=13)
ax.set_xlabel("Distance Traveled (x)")
ax.set_ylabel("Cross-track (y)")
ax.set_zlabel("Signal Level / Battery Voltage")
plt.colorbar(surf, shrink=0.5)
plt.show()

print("تهانينا! لقد أتممت اللاب 5 وأتقنت المؤثرات غير المتجانسة والتخميد الأسي.")
